# AH | Preparación y modelo dimensional

**Autor:** Alan Arturo Hernández Gutiérrez.

Este cuaderno ejecuta el ETL, comprueba reglas de frontera y valida el modelo. Exporta dos hechos y cuatro dimensiones como CSV; otras tres dimensiones se calculan en Power BI. Las operaciones documentadas están en los cuadernos de `src/` para conservar una sola implementación.

Se usa el ZIP autorizado de `input/` si existe; también puede indicar `AH_SOURCE_ZIP` o una ruta al ejecutar. El contexto municipal ITER es opcional mediante `AH_ITER_ZIP`; si falta, sus campos permanecen nulos.


In [ ]:
from pathlib import Path
import os,sys,json,contextlib,io
import pandas as pd
root=Path.cwd()
if root.name == "notebooks": root=root.parent
if not (root/"src"/"etl.ipynb").exists(): raise RuntimeError("Abrir desde AH_Adopcion_Digital o notebooks")
(root/"evidencia").mkdir(exist_ok=True)
if not os.environ.get("AH_SOURCE_ZIP"):
    local = root / "input" / "Tablas_Dimensiones_y_Ventas.zip"
    ruta = str(local) if local.is_file() else input("Ruta local del ZIP fuente autorizado: ").strip().strip('"')
    if not Path(ruta).is_file():
        raise FileNotFoundError("Seleccione un ZIP fuente existente")
    os.environ["AH_SOURCE_ZIP"] = ruta
cuaderno_etl = root / "src" / "etl.ipynb"
%run "$cuaderno_etl"
cuaderno_pruebas = root / "src" / "pruebas_etl.ipynb"
%run "$cuaderno_pruebas"
print("Entorno y rutas de proyecto preparados.")

## 1. Reglas y parámetros

Una fila de hechos es ticket–SKU. El snapshot es cliente–corte–regla. Las preferencias utilizan fechas distintas, importe, unidades y desempate estable; el antecedente digital excluye el día del primer Internet. Las cohortes inmaduras conservan nulos, no ceros. La moneda no está acreditada.

In [ ]:
config=json.loads((root/"config"/"proyecto.json").read_text(encoding="utf-8"))
print(json.dumps({k:v for k,v in config.items() if k!="archivo_iter_local"},ensure_ascii=False,indent=2))

## 2. Probar fronteras antes de procesar

Pruebas sintéticas identificadas como tales: visitas vs líneas, antecedentes, madurez, empate de canal, desconocidos, corte anterior, preferencias, precisión monetaria y total municipal ITER. Sus datos nunca se incorporan al modelo observado.

In [ ]:
pruebas=run_tests()
print(pruebas["detalle"])
(root/"evidencia"/"pruebas_etl.json").write_text(json.dumps(pruebas,ensure_ascii=False,indent=2),encoding="utf-8")

## 3. Ejecutar la preparación, enriquecimiento y modelo

Los cuadernos auxiliares src/etl.ipynb y src/enriquecimiento.ipynb contienen la implementación completa distribuida con este notebook. No se duplica la lógica en cada celda. El ETL verifica hashes, claves, cobertura y totales; construye primero en un directorio temporal y solo promueve una salida validada. Una reejecución completa no añade filas al resultado anterior. No ejecutar dos procesos simultáneamente.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()) as registro:
    resultado=run(os.environ["AH_SOURCE_ZIP"])
print(registro.getvalue())
actual=root/"evidencia"/"modelo_actual"
esquema=json.loads((actual/"auditoria"/"EsquemaModelo.json").read_text(encoding="utf-8"))
print(pd.DataFrame([{"Tabla":k,"Filas":v["rows"]} for k,v in esquema.items()]).to_string(index=False))

## 4. Revisar auditoría e integridad

Las comprobaciones del modelo se ejecutan sobre la construcción y se releen los seis CSV exportados para comprobar filas, columnas y centavos. Las 12 pruebas de fronteras anteriores son un control separado. No equivalen a ejecución de DAX o filtros en Desktop.

In [ ]:
validaciones=pd.read_csv(actual/"auditoria"/"Validaciones.csv")
print(validaciones.groupby("Resultado").size().to_string())
assert validaciones.Resultado.eq("OK").all()
manifest=json.loads((actual/"auditoria"/"MANIFIESTO.json").read_text(encoding="utf-8"))
for relativo,spec in manifest["archivos"].items():
    assert sha256_file(actual/relativo)==spec["sha256"],relativo
print("Archivos del manifiesto releídos y hashes comprobados.")
print(json.dumps(manifest["enriquecimiento"],ensure_ascii=False,indent=2))

## 5. Resultados de referencia para Power BI

Adopción a 90 días usa solo clientes elegibles con seguimiento completo. No es conversión web ni efecto de campaña. La concentración utiliza importes positivos y no margen o rentabilidad.

In [ ]:
sensibilidad=pd.read_csv(actual/"analisis"/"SensibilidadReglas.csv")
print(sensibilidad.to_string(index=False))
cohortes=pd.read_csv(actual/"analisis"/"Cohortes90.csv")
print(cohortes.loc[cohortes.ReglaKey.eq(1)].to_string(index=False))

In [ ]:
cuaderno_graficos = root / "src" / "graficos.ipynb"
%run "$cuaderno_graficos"
from IPython.display import display,Image
for figura in model_results(actual): display(Image(filename=str(figura)))

## Traslado a Power BI

Los seis CSV de `output/` son las tablas importadas en Power BI. `DimFecha`, `DimCorte` y `DimTiendaPreferida` se crean en DAX desde las tablas importadas. El diccionario de KPI y la arquitectura documentan medidas y relaciones. La carga y las cifras deberán comprobarse en Power BI Desktop.
